In [1]:
import os
import re
import glob
import joblib
import numpy as np
from scipy.stats import skew, kurtosis
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

# 1. File paths
DATA_DIR = r"C:\Users\simra\Desktop\walking_model\walking_dataset"
MODEL_SAVE_PATH = r"C:\Users\simra\Desktop\walking_model\final_gait_pd_model.pkl"

# 2. Feature extraction function (56 clinical & spectral features per 5-sec window)
def extract_window_features(win):
    left_s  = win[:, 0:8]
    right_s = win[:, 8:16]
    left_t  = win[:, 16]
    right_t = win[:, 17]
    
    feats = []
    for tot, sens in [(left_t, left_s), (right_t, right_s)]:
        on_ground = (tot > 0.12).astype(int)
        diffs = np.diff(on_ground)
        hs = np.flatnonzero(diffs == 1)
        
        if len(hs) >= 2:
            st = np.diff(hs) / 100.0
            s_mean, s_std = float(np.mean(st)), float(np.std(st))
            s_cv = (s_std / (s_mean + 1e-8)) * 100.0
        else:
            s_mean, s_std, s_cv = 1.1, 0.05, 4.5
            
        stance_ratio = float(np.mean(on_ground))
        heel_f = float(np.mean(sens[:, 0:2]))
        toe_f  = float(np.mean(sens[:, 6:8]))
        ht_ratio = heel_f / (toe_f + 1e-8)
        
        # FFT walk band (0.5-3 Hz) vs freeze/tremor band (3-8 Hz)
        fft_v = np.abs(np.fft.rfft(tot - np.mean(tot)))
        freqs = np.fft.rfftfreq(len(tot), d=0.01)
        w_en = float(np.sum(fft_v[(freqs >= 0.5) & (freqs <= 3.0)] ** 2))
        f_en = float(np.sum(fft_v[(freqs > 3.0) & (freqs <= 8.0)] ** 2))
        freeze_idx = f_en / (w_en + 1e-8)
        
        feats.extend([
            s_mean, s_std, s_cv, stance_ratio, ht_ratio, freeze_idx,
            float(np.max(tot)), float(np.std(tot)), float(skew(tot)), float(kurtosis(tot))
        ])
        feats.extend([float(x) for x in np.mean(sens, axis=0)])
        feats.extend([float(x) for x in np.std(sens, axis=0)])
        
    # Bilateral coordination & asymmetry features
    lr_corr = float(np.corrcoef(left_t, right_t).item(1))
    double_supp = float(np.mean((left_t > 0.12) & (right_t > 0.12)))
    asym_stride = abs(feats.__getitem__(0) - feats.__getitem__(26))
    asym_stance = abs(feats.__getitem__(3) - feats.__getitem__(29))
    
    feats.extend([lr_corr, double_supp, asym_stride, asym_stance])
    return feats

In [2]:
# 3. Load all files and build windowed dataset
print("Step 1/3: Loading walking files and extracting 5-second windows...")
X_win, y_win, pid_win = [], [], []

for file in sorted(glob.glob(os.path.join(DATA_DIR, "*.txt"))):
    fname = os.path.basename(file)
    match = re.match(r"^(Ga|Ju|Si)(Co|Pt)(\d+)_.*\.txt$", fname)
    if not match:
        continue
    study, group, num = match.groups()
    pid = f"{study}{group}{num}"
    label = 1 if group == "Pt" else 0
    
    raw = np.loadtxt(file)
    if len(raw) < 1600:
        continue
        
    sensors = raw[500:-500, 1:19]
    scale = np.mean(sensors[:, 16] + sensors[:, 17]) + 1e-8
    sensors = sensors / scale
    
    for start in range(0, len(sensors) - 500, 250):
        X_win.append(extract_window_features(sensors[start : start + 500]))
        y_win.append(label)
        pid_win.append(pid)

X_win = np.nan_to_num(np.array(X_win, dtype=np.float32))
y_win = np.array(y_win)
pid_win = np.array(pid_win)
print(f"Loaded {len(X_win)} windows across {len(np.unique(pid_win))} unique people.\n")


Step 1/3: Loading walking files and extracting 5-second windows...
Loaded 11596 windows across 165 unique people.



In [3]:
# 4. 5-Fold Cross-Validation (Prints progress for every Fold & Boosting Round)
print("Step 2/3: Running 5-Fold Person-Level Evaluation...")
cv = StratifiedGroupKFold(n_splits=5)
all_true, all_pred, all_prob = [], [], []

for fold, (tr, te) in enumerate(cv.split(X_win, y_win, pid_win), 1):
    print(f"\n--- Training Fold {fold}/5 (Train windows: {len(tr)} | Test windows: {len(te)}) ---")
    
    xgb = XGBClassifier(
        n_estimators=300, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
        n_jobs=-1, random_state=42
    )
    # Fit XGBoost separately first so you can see boosting round progress every 100 rounds
    xgb.fit(X_win[tr], y_win[tr], eval_set=[(X_win[te], y_win[te])], verbose=100)
    
    et = ExtraTreesClassifier(n_estimators=300, max_depth=16, n_jobs=-1, random_state=42)
    rf = RandomForestClassifier(n_estimators=300, max_depth=14, n_jobs=-1, random_state=42)
    
    ensemble = VotingClassifier([('xgb', xgb), ('et', et), ('rf', rf)], voting='soft')
    ensemble.fit(X_win[tr], y_win[tr])
    
    probs = ensemble.predict_proba(X_win[te])[:, 1]
    fold_true, fold_pred = [], []
    
    for pid in np.unique(pid_win[te]):
        mask = (pid_win[te] == pid)
        avg_prob = float(np.mean(probs[mask]))
        pred_lbl = 1 if avg_prob > 0.5 else 0
        true_lbl = int(y_win[te][mask].item(0))
        
        fold_prob = avg_prob
        fold_pred.append(pred_lbl)
        fold_true.append(true_lbl)
        
        all_prob.append(avg_prob)
        all_pred.append(pred_lbl)
        all_true.append(true_lbl)
        
    print(f"Fold {fold}/5 Person-Level Accuracy: {accuracy_score(fold_true, fold_pred) * 100:.2f}%")


Step 2/3: Running 5-Fold Person-Level Evaluation...

--- Training Fold 1/5 (Train windows: 9286 | Test windows: 2310) ---
[0]	validation_0-logloss:0.60535
[100]	validation_0-logloss:0.54459
[200]	validation_0-logloss:0.56338
[299]	validation_0-logloss:0.61770
Fold 1/5 Person-Level Accuracy: 70.97%

--- Training Fold 2/5 (Train windows: 9277 | Test windows: 2319) ---
[0]	validation_0-logloss:0.60535
[100]	validation_0-logloss:0.71586
[200]	validation_0-logloss:0.84944
[299]	validation_0-logloss:0.92331
Fold 2/5 Person-Level Accuracy: 78.12%

--- Training Fold 3/5 (Train windows: 9269 | Test windows: 2327) ---
[0]	validation_0-logloss:0.60385
[100]	validation_0-logloss:0.55997
[200]	validation_0-logloss:0.65565
[299]	validation_0-logloss:0.73432
Fold 3/5 Person-Level Accuracy: 76.47%

--- Training Fold 4/5 (Train windows: 9276 | Test windows: 2320) ---
[0]	validation_0-logloss:0.60549
[100]	validation_0-logloss:0.62409
[200]	validation_0-logloss:0.63556
[299]	validation_0-logloss:0.66274

In [4]:
# 5. Print Complete Evaluation Summary Across All 165 People
print("\n" + "="*55)
print("FINAL GAIT MODEL EVALUATION REPORT (ALL 165 PEOPLE)")
print("="*55)
print(f"Overall Person-Level Accuracy : {accuracy_score(all_true, all_pred) * 100:.2f}%")
print(f"ROC-AUC Score                 : {roc_auc_score(all_true, all_prob):.4f}\n")
print("Confusion Matrix (Rows = Actual, Cols = Predicted):")
print(confusion_matrix(all_true, all_pred))
print("\nDetailed Classification Report:")
print(classification_report(all_true, all_pred, target_names=["Healthy (Co)", "Parkinson's (Pt)"]))


FINAL GAIT MODEL EVALUATION REPORT (ALL 165 PEOPLE)
Overall Person-Level Accuracy : 75.76%
ROC-AUC Score                 : 0.8312

Confusion Matrix (Rows = Actual, Cols = Predicted):
[[50 22]
 [18 75]]

Detailed Classification Report:
                  precision    recall  f1-score   support

    Healthy (Co)       0.74      0.69      0.71        72
Parkinson's (Pt)       0.77      0.81      0.79        93

        accuracy                           0.76       165
       macro avg       0.75      0.75      0.75       165
    weighted avg       0.76      0.76      0.76       165



In [6]:
# 6. Train Final Production Model on 100% of Data & Save to .pkl
print("="*55)
print("Step 3/3: Training Final Model on 100% Data & Saving...")
final_gait_model = VotingClassifier([
    ('xgb', XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42)),
    ('et',  ExtraTreesClassifier(n_estimators=300, max_depth=12, n_jobs=-1, random_state=42)),
    ('rf',  RandomForestClassifier(n_estimators=300, max_depth=12, n_jobs=-1, random_state=42))
], voting='soft')

final_gait_model.fit(X_win, y_win)
joblib.dump(final_gait_model, MODEL_SAVE_PATH, compress=3)
print(f"DONE! Model successfully stored at:\n{MODEL_SAVE_PATH}")


Step 3/3: Training Final Model on 100% Data & Saving...
DONE! Model successfully stored at:
C:\Users\simra\Desktop\walking_model\final_gait_pd_model.pkl
